## Modeling

0. Index
1. Train the models
2. Evaluate the models
3. Compare models

### 1. Train the model
goal: 
    determine the best_params, cv_score and val_score of each model

In [1]:
import pandas as pd
from pathlib import Path
import os
import sys

In [2]:
# This adds the parent directory (project root) to the Python path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

In [3]:
from src.models.train import train_model

In [4]:
X_train_fe = pd.read_csv('../data/processed/X_train.csv')
X_val_fe = pd.read_csv('../data/processed/X_val.csv')
X_test_fe = pd.read_csv('../data/processed/X_test.csv')

Y_train = pd.read_csv('../data/processed/Y_train.csv').squeeze()
Y_val = pd.read_csv('../data/processed/Y_val.csv').squeeze()
Y_test = pd.read_csv('../data/processed/Y_test.csv').squeeze()

In [5]:
X_train_fe.head()

,sex,cp,thal,oldpeak,ca
0,1.0,0.0,3.0,2.6,0.0
1,1.0,0.0,3.0,0.0,0.0
2,1.0,0.0,3.0,0.0,2.0
3,1.0,0.0,3.0,1.0,2.0
4,1.0,0.0,3.0,0.0,0.0


In [6]:
print("Shapes: ")
print(f" X_train:  {X_train_fe.shape}")
print(f" X_val:  {X_val_fe.shape}")
print(f" X_test:  {X_test_fe.shape}")

Shapes: 
 X_train:  (180, 5)
 X_val:  (61, 5)
 X_test:  (61, 5)


In [7]:
result_lr = train_model( "logistic_regression", X_train_fe, Y_train, X_val_fe ,Y_val)
result_rf = train_model( "random_forest", X_train_fe, Y_train, X_val_fe ,Y_val)
result_xgb = train_model( "xgboost", X_train_fe, Y_train, X_val_fe ,Y_val)

In [8]:
result_lr

{'model': LogisticRegression(C=10.0, max_iter=1000, random_state=42),
 'scaler': StandardScaler(),
 'best_params': {'C': 10.0},
 'cv_score': np.float64(0.9426973684210527),
 'val_score': 0.783008658008658}

In [9]:
result_rf

{'model': RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=200,
                        n_jobs=1, random_state=42),
 'scaler': None,
 'best_params': {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200},
 'cv_score': np.float64(0.9470201238390092),
 'val_score': 0.7862554112554113}

In [10]:
result_xgb

{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.05, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=2, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=100, n_jobs=None,
               num_parallel_tree=None, ...),
 'scaler': None,
 'best_params': {'learning_rate': 0.05,
  'max_depth': 2,
  'n_estimators': 100,
  'subsample': 0.8},
 'cv_score': np.float64(0.9560826238390092),
 'val_score': 0.7927489177489178}

### 2. Evaluate the model
goal: 
    calculate indicators such as Accuracy, ROC-AUC, Precision (1), Recall, F1 ,CV ROC-AUC and Gap of each model, and determine which will go to production


In [11]:
from src.models.evaluate import evaluate_model

In [12]:
metrics_lr = evaluate_model( 
    "logistic_regression", 
    result_lr["model"], 
    X_test_fe, 
    Y_test, 
    result_lr["cv_score"], 
    result_lr["scaler"]
    )


───────────────────────────────────────────────────────
logistic_regression

───────────────────────────────────────────────────────
 Accuracy : 0.8197
 ROC-AUC : 0.8815
 Precision : 0.8438
 Recall : 0.8182
 F1 : 0.8308
 CV ROC-AUC : 0.9427 (train)
 Gap : 0.0612 -> possible overfitting

  Confusion Matrix:
    TN (no disease, correct)  : 23
    FP (no disease, wrong)    : 5  ← predicted disease, was healthy
    FN (disease, missed)      : 6  ← predicted healthy, had disease
    TP (disease, correct)     : 27

              precision    recall  f1-score   support

  No Disease       0.79      0.82      0.81        28
     Disease       0.84      0.82      0.83        33

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



In [13]:
metrics_rf = evaluate_model( 
    "random_forest", 
    result_rf["model"],
    X_test_fe, 
    Y_test, 
    result_rf["cv_score"], 
    result_rf["scaler"]
    )


───────────────────────────────────────────────────────
random_forest

───────────────────────────────────────────────────────
 Accuracy : 0.8033
 ROC-AUC : 0.8858
 Precision : 0.8000
 Recall : 0.8485
 F1 : 0.8235
 CV ROC-AUC : 0.9470 (train)
 Gap : 0.0612 -> possible overfitting

  Confusion Matrix:
    TN (no disease, correct)  : 21
    FP (no disease, wrong)    : 7  ← predicted disease, was healthy
    FN (disease, missed)      : 5  ← predicted healthy, had disease
    TP (disease, correct)     : 28

              precision    recall  f1-score   support

  No Disease       0.81      0.75      0.78        28
     Disease       0.80      0.85      0.82        33

    accuracy                           0.80        61
   macro avg       0.80      0.80      0.80        61
weighted avg       0.80      0.80      0.80        61



In [14]:
metrics_xgb = evaluate_model(
     "xgboost", 
     result_xgb["model"], 
     X_test_fe, 
     Y_test, 
     result_xgb["cv_score"], 
     result_xgb["scaler"])


───────────────────────────────────────────────────────
xgboost

───────────────────────────────────────────────────────
 Accuracy : 0.7869
 ROC-AUC : 0.8734
 Precision : 0.7778
 Recall : 0.8485
 F1 : 0.8116
 CV ROC-AUC : 0.9561 (train)
 Gap : 0.0827 -> possible overfitting

  Confusion Matrix:
    TN (no disease, correct)  : 20
    FP (no disease, wrong)    : 8  ← predicted disease, was healthy
    FN (disease, missed)      : 5  ← predicted healthy, had disease
    TP (disease, correct)     : 28

              precision    recall  f1-score   support

  No Disease       0.80      0.71      0.75        28
     Disease       0.78      0.85      0.81        33

    accuracy                           0.79        61
   macro avg       0.79      0.78      0.78        61
weighted avg       0.79      0.79      0.79        61



### 3. Compare the model
goal: 
    table to show the performance of the models

In [15]:
from src.models.compare import compare_models

In [16]:
all_metrics = [metrics_lr, metrics_rf, metrics_xgb]
all_results = [result_lr, result_rf, result_xgb]

comparison_df = compare_models(all_metrics, all_results, X_test_fe, Y_test)


  MODEL COMPARISON — TEST SET
                     Accuracy  ROC-AUC  Precision  Recall      F1  CV Score     Gap
model                                                                              
random_forest          0.8033   0.8858     0.8000  0.8485  0.8235    0.9470  0.0612
logistic_regression    0.8197   0.8815     0.8438  0.8182  0.8308    0.9427  0.0612
xgboost                0.7869   0.8734     0.7778  0.8485  0.8116    0.9561  0.0827

  Best model by ROC-AUC: random_forest
  ✔ ROC curves saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/roc_curves_all_models.png


In [17]:
results_map = {
    metrics_lr["model_name"]: result_lr,
    metrics_rf["model_name"]: result_rf,
    metrics_xgb["model_name"]: result_xgb,
}

metrics_map = {
    metrics_lr["model_name"]: metrics_lr,
    metrics_rf["model_name"]: metrics_rf,
    metrics_xgb["model_name"]: metrics_xgb,
}

In [18]:
# Identify the best model
best_idx     = comparison_df["ROC-AUC"].idxmax()
best_result  = results_map[best_idx]
best_metrics = metrics_map[best_idx]
best_result

{'model': RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=200,
                        n_jobs=1, random_state=42),
 'scaler': None,
 'best_params': {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200},
 'cv_score': np.float64(0.9470201238390092),
 'val_score': 0.7862554112554113}

In [19]:
print(f"Best model: {best_idx}")

Best model: random_forest


### 4. Explain results


In [20]:
from src.models.explain import explain_model

/Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
X_test_in = (
    pd.DataFrame(best_result["scaler"].transform(X_test_fe), columns=X_test_fe.columns, index=X_test_fe.index)
    if best_result["scaler"] is not None
    else X_test_fe
)

shap_importance = explain_model(
    best_result["model"],
    X_test_in,
    Y_test
)

  Generating SHAP bar chart (global importance)...
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_bar.png
  Generating SHAP beeswarm plot...
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_beeswarm.png

  SHAP Importance Table:
feature  mean_abs_shap share
     cp       0.179662 29.7%
   thal       0.148163 24.5%
     ca       0.137705 22.8%
oldpeak       0.104614 17.3%
    sex       0.033848  5.6%

  Top feature 'cp' accounts for 29.7% of total SHAP importance.

  Force plot — patient 0:
    True label     : 0
    Predicted      : 0
    P(disease)     : 6.02%
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_force_patient_0.png


In [22]:
shap_importance

,feature,mean_abs_shap,share
0,cp,0.179662,29.7%
1,thal,0.148163,24.5%
2,ca,0.137705,22.8%
3,oldpeak,0.104614,17.3%
4,sex,0.033848,5.6%


In [23]:
from importlib.metadata import version

In [24]:
version("xgboost")  

'3.3.0'

In [25]:
from src.utils.helpers import save_pickle, save_json
from src.utils.config import  CFG, PROJECT_ROOT, RISK_BANDS
import datetime

In [26]:
now = datetime.datetime.now()
date_string = now.strftime("%m.%d.%y")
date_string

'08.01.26'

In [28]:
metadata = {
    "model_name": best_idx,
    "model_version": date_string,
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "decision_threshold": CFG["evaluation"]["decision_threshold"],
    "features": CFG["features"]["selected"],
    "metrics": {
        "roc_auc": best_metrics["roc-auc"],
        "recall": best_metrics["recall"],
        "accuracy": best_metrics["accuracy"],
        "precision": best_metrics["precision"],
        "f1-score": best_metrics["f1"],
    },
    "shap_importance": shap_importance[["feature", "share"]].to_dict(orient="records"),
    "libraries":{
        "scikit-learn": "sklearn.1.9.0",
    },
    "author": "Jhoselyn Pajuelo Villanueva", 
    "probability":  RISK_BANDS

}

In [29]:
metadata

{'model_name': 'random_forest',
 'model_version': '08.01.26',
 'trained_at': '2026-08-01T17:12:59.446463+00:00',
 'decision_threshold': 0.5,
 'features': ['sex', 'cp', 'thal', 'oldpeak', 'ca'],
 'metrics': {'roc_auc': 0.8858225108225108,
  'recall': 0.8484848484848485,
  'accuracy': 0.8032786885245902,
  'precision': 0.8,
  'f1-score': 0.8235294117647058},
 'shap_importance': [{'feature': 'cp', 'share': '29.7%'},
  {'feature': 'thal', 'share': '24.5%'},
  {'feature': 'ca', 'share': '22.8%'},
  {'feature': 'oldpeak', 'share': '17.3%'},
  {'feature': 'sex', 'share': '5.6%'}],
 'libraries': {'scikit-learn': 'sklearn.1.9.0'},
 'author': 'Jhoselyn Pajuelo Villanueva',
 'probability': [{'level': 'LOW',
   'label': 'Low Risk',
   'max': 0.3,
   'recommendations': ['Continue routine cardiovascular health maintenance and periodic screening.',
    'Reinforce heart-healthy lifestyle measures (diet, activity, smoking cessation).',
    'Monitor blood pressure and lipid profile at routine follow-up 

In [30]:
save_json(metadata, PROJECT_ROOT / "models" / "model_metadata.json")

  ✔ Saved  → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/models/model_metadata.json


In [31]:
save_pickle(best_result["model"],  PROJECT_ROOT / "models" / "best_model.pkl")
#save_pickle(best_result["scaler"], PROJECT_ROOT / "models" / "scaler.pkl")  - if LR is best model

saved /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/models/best_model.pkl
